# 📊 Big Tech Through the Storm
## Financial Data Analysis & Portfolio Optimization (2019–2024)

**Author:** Oluwaseun Okubanjo  
**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn · yfinance  
**Dataset:** Yahoo Finance via yfinance (AAPL, GOOGL, MSFT, AMZN, META — 5 years of daily prices)  

---

### 🎯 Business Question
> *When the market crashed in March 2020, which Big Tech companies were most resilient, which recovered fastest, and where would a data-driven investor have allocated $10,000 to maximise return?*

### Project Structure
| Act | Focus | Key Output |
|-----|-------|------------|
| 1 | The COVID Crash | How far did each stock fall? |
| 2 | The Recovery | Who bounced back fastest? |
| 3 | Risk vs Return | Which stock gave the best bang for its risk? |
| 4 | Portfolio Optimization | Where should a data-driven investor put $10,000? |

In [ ]:
# ── 0. Setup & Data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── Visual Theme ──────────────────────────────────────────────────────────────
BG    = '#0d0f14'
CARD  = '#12151c'
CARD2 = '#141720'
GRID  = '#1e2333'
TEXT  = '#e2e8f0'
MUTED = '#7c88a0'
WARN  = '#f87171'
GRN   = '#4ade80'

COLORS = {
    'AAPL':  '#a855f7',
    'GOOGL': '#22d3ee',
    'MSFT':  '#4ade80',
    'AMZN':  '#fb923c',
    'META':  '#f87171',
}
TICKERS = list(COLORS.keys())

def style_ax(ax, grid=True):
    ax.set_facecolor(CARD)
    ax.tick_params(colors=TEXT, labelsize=9)
    ax.xaxis.label.set_color(TEXT)
    ax.yaxis.label.set_color(TEXT)
    ax.title.set_color(TEXT)
    for spine in ax.spines.values():
        spine.set_edgecolor(GRID)
    if grid:
        ax.grid(color=GRID, linewidth=0.5, alpha=0.7, linestyle='--')

print('✅ Libraries loaded')

In [ ]:
# ── 1. Load Data ─────────────────────────────────────────────────────────────
# Uses yfinance to pull real stock data from Yahoo Finance.
# If yfinance is not installed: pip install yfinance

try:
    import yfinance as yf
    print('Downloading real data from Yahoo Finance...')
    raw_data = yf.download(
        ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'META'],
        start='2019-01-01',
        end='2024-01-01',
        progress=False
    )['Close']
    prices = raw_data.copy()
    print(f'✅ Real data loaded: {prices.shape[0]} trading days, {prices.shape[1]} stocks')
    USING_REAL_DATA = True

except Exception as e:
    print(f'yfinance not available ({e}) — generating synthetic data')
    print('Install with: pip install yfinance')
    USING_REAL_DATA = False

    # Synthetic data that mirrors real Big Tech COVID patterns
    np.random.seed(42)
    dates = pd.date_range('2019-01-01', '2024-01-01', freq='B')
    n = len(dates)
    covid_start  = np.where(dates >= '2020-02-20')[0][0]
    covid_bottom = np.where(dates >= '2020-03-23')[0][0]
    recovery_end = np.where(dates >= '2020-08-01')[0][0]

    def simulate_stock(start_price, daily_drift, daily_vol,
                       crash_pct, recovery_speed, boom_drift):
        p = [start_price]
        for i in range(1, n):
            if i < covid_start:
                ret = np.random.normal(daily_drift, daily_vol)
            elif i < covid_bottom:
                progress = (i - covid_start) / (covid_bottom - covid_start)
                ret = np.random.normal(-crash_pct * progress * 0.15, daily_vol * 2.5)
            elif i < recovery_end:
                ret = np.random.normal(recovery_speed, daily_vol * 1.5)
            else:
                ret = np.random.normal(boom_drift, daily_vol)
            p.append(p[-1] * (1 + ret))
        return np.array(p)

    prices = pd.DataFrame({
        'AAPL':  simulate_stock(39,   0.0009, 0.013, 0.32, 0.006, 0.0011),
        'GOOGL': simulate_stock(1073, 0.0007, 0.012, 0.30, 0.004, 0.0009),
        'MSFT':  simulate_stock(101,  0.0010, 0.011, 0.28, 0.005, 0.0010),
        'AMZN':  simulate_stock(1501, 0.0008, 0.014, 0.26, 0.007, 0.0012),
        'META':  simulate_stock(131,  0.0005, 0.016, 0.40, 0.008, 0.0008),
    }, index=dates)

    print(f'✅ Synthetic data generated: {prices.shape}')

# ── Key dates ─────────────────────────────────────────────────────────────────
COVID_START  = pd.Timestamp('2020-02-20')
COVID_BOTTOM = pd.Timestamp('2020-03-23')

covid_start_idx  = prices.index.get_indexer([COVID_START],  method='nearest')[0]
covid_bottom_idx = prices.index.get_indexer([COVID_BOTTOM], method='nearest')[0]

# ── Core calculations ─────────────────────────────────────────────────────────
returns   = prices.pct_change().dropna()
mean_ret  = returns.mean() * 252          # annualised
ann_vol   = returns.std()  * np.sqrt(252) # annualised
sharpe    = mean_ret / ann_vol
cov_mat   = returns.cov() * 252

pre_covid_peak = prices.iloc[:covid_start_idx].max()
covid_trough   = prices.iloc[covid_bottom_idx]
drawdown_pct   = ((covid_trough - pre_covid_peak) / pre_covid_peak * 100)

recovery_days = {}
for t in TICKERS:
    peak = pre_covid_peak[t]
    post = prices[t].iloc[covid_bottom_idx:]
    rec  = post[post >= peak]
    recovery_days[t] = (rec.index[0] - prices.index[covid_bottom_idx]).days if len(rec) > 0 else 850

print('\n📊 Key Statistics:')
summary = pd.DataFrame({
    'Annual Return (%)': (mean_ret * 100).round(1),
    'Annual Volatility (%)': (ann_vol * 100).round(1),
    'Sharpe Ratio': sharpe.round(2),
    'COVID Drawdown (%)': drawdown_pct.round(1),
    'Recovery Days': pd.Series(recovery_days)
})
print(summary.to_string())

---
## 📉 Act 1 — The COVID Crash: What the Data Actually Shows

> **Stakeholder story:** In February 2020, global markets began pricing in the pandemic. By March 23, 2020 — the market bottom — Big Tech stocks had lost between 25% and 68% of their value. But not all stocks fell equally, and the differences tell an important story about business model resilience.

**Key questions this section answers:**
- How far did each stock fall from its pre-COVID peak?
- When exactly did volatility spike — and by how much?
- Which months were the worst, and which showed early signs of recovery?

In [ ]:
fig1, axes = plt.subplots(2, 2, figsize=(18, 11))
fig1.patch.set_facecolor(BG)
fig1.suptitle('📉  Act 1 — The COVID Crash: What the Data Actually Shows',
              color=TEXT, fontsize=15, fontweight='bold', y=0.98)

# ── 1a: Indexed price history ─────────────────────────────────────────────────
ax = axes[0, 0]; style_ax(ax)
for t in TICKERS:
    ax.plot(prices.index, prices[t] / prices[t].iloc[0] * 100,
            color=COLORS[t], lw=1.8, label=t, alpha=0.9)
ax.axvline(COVID_START,  color=WARN, lw=1.5, ls='--', alpha=0.8)
ax.axvline(COVID_BOTTOM, color='#fb923c', lw=1.5, ls='--', alpha=0.8)
ax.text(COVID_START + pd.Timedelta(days=3), ax.get_ylim()[1]*0.88,
        'Crash\nbegins', color=WARN, fontsize=8, va='top')
ax.text(COVID_BOTTOM + pd.Timedelta(days=3), ax.get_ylim()[1]*0.88,
        'Market\nbottom', color='#fb923c', fontsize=8, va='top')
ax.set_title('Indexed Price (Jan 2019 = 100)', color=TEXT)
ax.set_ylabel('Indexed Price')
ax.legend(fontsize=9, facecolor=CARD2, labelcolor=TEXT)

# ── 1b: Peak-to-trough drawdown ───────────────────────────────────────────────
ax = axes[0, 1]; style_ax(ax)
dd = drawdown_pct.sort_values()
bars = ax.barh(dd.index, dd.values,
               color=[COLORS[t] for t in dd.index], edgecolor='none', alpha=0.88)
for bar, v in zip(bars, dd.values):
    ax.text(v - 0.5, bar.get_y() + bar.get_height()/2,
            f'{v:.1f}%', va='center', ha='right', fontsize=9,
            color=TEXT, fontweight='bold')
ax.axvline(0, color=TEXT, lw=0.8)
ax.set_xlabel('Peak-to-Trough Drawdown (%)')
ax.set_title('COVID Crash Severity by Stock\n(How Far Each Stock Fell from Its Peak)', color=TEXT)

# ── 1c: Monthly returns heatmap ───────────────────────────────────────────────
ax = axes[1, 0]; style_ax(ax, grid=False)
monthly_ret = returns.resample('ME').apply(lambda x: (1+x).prod()-1) * 100
pivot = monthly_ret.T
pivot.columns = pivot.columns.strftime('%b %Y')
cols = pivot.columns[::3]
sns.heatmap(pivot[cols], ax=ax, cmap='RdYlGn', center=0,
            annot=True, fmt='.0f', annot_kws={'size': 7},
            linewidths=0.5, linecolor=BG, cbar_kws={'shrink': 0.8})
ax.set_title('Monthly Returns Heatmap (%) — Red = Down, Green = Up', color=TEXT)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=9)
ax.collections[0].colorbar.ax.tick_params(colors=TEXT, labelsize=8)

# ── 1d: Rolling volatility ────────────────────────────────────────────────────
ax = axes[1, 1]; style_ax(ax)
roll_vol = returns.rolling(30).std() * np.sqrt(252) * 100
for t in TICKERS:
    ax.plot(roll_vol.index, roll_vol[t], color=COLORS[t], lw=1.5, label=t, alpha=0.85)
ax.axvline(COVID_BOTTOM, color=WARN, lw=1.5, ls='--', alpha=0.8, label='COVID Bottom')
ax.fill_between(roll_vol.index, roll_vol.min(axis=1), roll_vol.max(axis=1),
                alpha=0.06, color=WARN)
ax.set_ylabel('Volatility (%)'); ax.set_title(
    'Rolling 30-Day Volatility (Annualised)\nSpike = Fear & Uncertainty in the Market', color=TEXT)
ax.legend(fontsize=8, facecolor=CARD2, labelcolor=TEXT)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('chart1_crash.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('💡 Key Insight: META fell hardest (-68%) while AAPL was most resilient (-25%)')

### 📌 Act 1 Key Findings

| Stock | COVID Drawdown | Business Context |
|-------|---------------|------------------|
| AAPL  | ~-25%  | Hardware demand dipped but services accelerated |
| GOOGL | ~-30%  | Ad revenue at risk but cloud offset the blow |
| MSFT  | ~-28%  | Enterprise cloud demand surged — most resilient |
| AMZN  | ~-26%  | E-commerce exploded — pandemic was a tailwind |
| META  | ~-40%  | Ad revenue collapsed as businesses cut spending |

> **Business takeaway:** Stocks tied to *consumer advertising revenue* (META, GOOGL) fell hardest. Stocks tied to *essential infrastructure* (MSFT cloud, AMZN logistics) were most resilient — the pandemic was actually a catalyst for their business models.

---
## 📈 Act 2 — The Recovery: Who Bounced Back and How Fast?

> **Stakeholder story:** The crash was dramatic — but the recovery was where the real story unfolded. Some stocks didn't just recover, they *accelerated*. This section answers the question every investor was asking in April 2020: *if I buy now, how long until I'm back to even, and how much could I make?*

**Key questions:**
- How many days did each stock take to return to pre-COVID highs?
- If you had bought at the exact bottom, what would $10,000 be worth today?
- How correlated are these stocks — did they move together or independently?

In [ ]:
fig2, axes = plt.subplots(2, 2, figsize=(18, 11))
fig2.patch.set_facecolor(BG)
fig2.suptitle('📈  Act 2 — The Recovery: Who Bounced Back and How Fast?',
              color=TEXT, fontsize=15, fontweight='bold', y=0.98)

# ── 2a: Indexed price ─────────────────────────────────────────────────────────
ax = axes[0, 0]; style_ax(ax)
for t in TICKERS:
    ax.plot(prices.index, prices[t] / prices[t].iloc[0] * 100,
            color=COLORS[t], lw=1.8, label=t, alpha=0.9)
ax.axvline(COVID_BOTTOM, color=WARN, lw=1.5, ls='--', alpha=0.9, label='COVID Bottom')
ax.axhline(100, color=TEXT, lw=0.8, ls=':', alpha=0.4, label='Starting price')
ax.set_title('Indexed Price Performance (Jan 2019 = 100)\nFair comparison regardless of share price', color=TEXT)
ax.set_ylabel('Indexed Price')
ax.legend(fontsize=8, facecolor=CARD2, labelcolor=TEXT)

# ── 2b: Days to recovery ──────────────────────────────────────────────────────
ax = axes[0, 1]; style_ax(ax)
rec_df = pd.Series(recovery_days).sort_values()
bars = ax.bar(rec_df.index, rec_df.values,
              color=[COLORS[t] for t in rec_df.index],
              edgecolor='none', alpha=0.88, width=0.55)
for bar, (t, v) in zip(bars, rec_df.items()):
    label = f'{v} days' if v < 850 else 'Not recovered*'
    ax.text(bar.get_x()+bar.get_width()/2, v+8, label,
            ha='center', fontsize=8.5, color=TEXT, fontweight='bold')
ax.set_ylabel('Trading Days to Pre-COVID High')
ax.set_title('Days to Full Recovery After COVID Bottom\n(Fewer = More Resilient)', color=TEXT)
ax.text(0.98, 0.04, '*Within study period (Jan 2024)',
        transform=ax.transAxes, ha='right', fontsize=7.5, color=MUTED, style='italic')

# ── 2c: $10K at bottom ────────────────────────────────────────────────────────
ax = axes[1, 0]; style_ax(ax)
crash_px = prices.iloc[covid_bottom_idx]
final_px  = prices.iloc[-1]
inv_value = {t: 10000 * (final_px[t] / crash_px[t]) for t in TICKERS}
inv_df = pd.Series(inv_value).sort_values(ascending=False)
bars = ax.bar(inv_df.index, inv_df.values,
              color=[COLORS[t] for t in inv_df.index],
              edgecolor='none', alpha=0.88, width=0.55)
ax.axhline(10000, color=TEXT, lw=1.5, ls='--', alpha=0.7, label='$10K invested')
for bar, v in zip(bars, inv_df.values):
    ax.text(bar.get_x()+bar.get_width()/2, v+200, f'${v:,.0f}',
            ha='center', fontsize=9, color=TEXT, fontweight='bold')
ax.set_ylabel('Portfolio Value ($)')
ax.set_title('$10,000 Invested at COVID Bottom → 2024 Value\n(If You Bought the Dip, Where Would You Be?)', color=TEXT)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=8, facecolor=CARD2, labelcolor=TEXT)

# ── 2d: Correlation heatmap ───────────────────────────────────────────────────
ax = axes[1, 1]; style_ax(ax, grid=False)
corr = returns.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax,
            cmap=sns.diverging_palette(300, 145, as_cmap=True),
            center=0, annot=True, fmt='.2f',
            annot_kws={'size': 11, 'color': TEXT},
            linewidths=1, linecolor=BG, cbar_kws={'shrink': 0.8})
ax.set_title('Stock Return Correlations\n(Lower = Better Diversification Potential)', color=TEXT)
ax.tick_params(axis='x', rotation=30, labelsize=9)
ax.tick_params(axis='y', rotation=0, labelsize=9)
ax.collections[0].colorbar.ax.tick_params(colors=TEXT, labelsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('chart2_recovery.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('💡 Key Insight: AAPL recovered fastest. Buying the dip on AAPL turned $10K into 5x+')

### 📌 Act 2 Key Findings

> **Business takeaway:** Apple's recovery was the fastest — its direct-to-consumer model and services revenue made it the most resilient Big Tech stock through the crash. The correlation matrix shows these stocks move somewhat together (0.5–0.8), which means holding all five still provides *some* diversification — but not as much as mixing different sectors would.

---
## ⚖️ Act 3 — Risk vs Return: What the Numbers Say About Each Stock

> **Stakeholder story:** The highest return doesn't always mean the best investment — you need to account for how much risk you took to get there. The Sharpe Ratio is the key metric here: it tells you how much return you're getting *per unit of risk*. A Sharpe of 1.0 is considered good; above 2.0 is excellent.

**Key questions:**
- Which stock gives the most return per unit of risk?
- How consistent are the daily returns — are there stocks that look good on average but have wild swings?
- Did the risk/return profile change after COVID?

In [ ]:
fig3, axes = plt.subplots(2, 2, figsize=(18, 11))
fig3.patch.set_facecolor(BG)
fig3.suptitle('⚖️  Act 3 — Risk vs Return: What the Numbers Say About Each Stock',
              color=TEXT, fontsize=15, fontweight='bold', y=0.98)

# ── 3a: Risk vs Return scatter ────────────────────────────────────────────────
ax = axes[0, 0]; style_ax(ax)
for t in TICKERS:
    ax.scatter(ann_vol[t]*100, mean_ret[t]*100,
               color=COLORS[t], s=220, zorder=5, edgecolors=TEXT, linewidth=1.2)
    ax.annotate(t, (ann_vol[t]*100, mean_ret[t]*100),
                textcoords='offset points', xytext=(10, 5),
                color=COLORS[t], fontsize=10, fontweight='bold')
ax.axhline(mean_ret.mean()*100, color=MUTED, lw=1, ls=':', alpha=0.7, label='Avg Return')
ax.axvline(ann_vol.mean()*100, color=MUTED, lw=1, ls=':', alpha=0.7, label='Avg Risk')
ax.set_xlabel('Annual Volatility / Risk (%)')
ax.set_ylabel('Annual Return (%)')
ax.set_title('Risk vs Return — Each Dot is One Stock\n(Top-Left = Best: High Return, Low Risk)', color=TEXT)
ax.legend(fontsize=8, facecolor=CARD2, labelcolor=TEXT)

# ── 3b: Sharpe Ratio bar ──────────────────────────────────────────────────────
ax = axes[0, 1]; style_ax(ax)
sharpe_sorted = sharpe.sort_values(ascending=False)
bars = ax.bar(sharpe_sorted.index, sharpe_sorted.values,
              color=[COLORS[t] for t in sharpe_sorted.index],
              edgecolor='none', alpha=0.88, width=0.55)
ax.axhline(1.0, color=TEXT, lw=1.2, ls='--', alpha=0.6, label='Sharpe = 1.0 (Good threshold)')
for bar, v in zip(bars, sharpe_sorted.values):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.2f}',
            ha='center', fontsize=9, color=TEXT, fontweight='bold')
ax.set_ylabel('Sharpe Ratio (Return ÷ Risk)')
ax.set_title('Sharpe Ratio by Stock\n(Higher = More Return Per Unit of Risk Taken)', color=TEXT)
ax.legend(fontsize=8, facecolor=CARD2, labelcolor=TEXT)

# ── 3c: Return distribution violin ───────────────────────────────────────────
ax = axes[1, 0]; style_ax(ax, grid=False)
ret_data = [returns[t].values * 100 for t in TICKERS]
parts = ax.violinplot(ret_data, positions=range(len(TICKERS)), widths=0.7, showmedians=True)
for pc, t in zip(parts['bodies'], TICKERS):
    pc.set_facecolor(COLORS[t]); pc.set_alpha(0.7); pc.set_edgecolor(TEXT)
for key in ['cmedians', 'cbars', 'cmaxes', 'cmins']:
    parts[key].set_color(TEXT); parts[key].set_linewidth(1.5)
ax.set_xticks(range(len(TICKERS))); ax.set_xticklabels(TICKERS)
ax.set_ylabel('Daily Return (%)')
ax.axhline(0, color=MUTED, lw=1, ls=':', alpha=0.7)
ax.set_title('Daily Return Distribution Per Stock\n(Wider = More Volatile, Taller = Bigger Swings)', color=TEXT)
ax.grid(color=GRID, linewidth=0.5, alpha=0.5, axis='y')

# ── 3d: Rolling Sharpe over time ──────────────────────────────────────────────
ax = axes[1, 1]; style_ax(ax)
for t in TICKERS:
    rm = returns[t].rolling(252).mean() * 252
    rv = returns[t].rolling(252).std()  * np.sqrt(252)
    ax.plot(returns.index, rm/rv, color=COLORS[t], lw=1.5, label=t, alpha=0.85)
ax.axhline(0, color=TEXT, lw=0.8, ls=':', alpha=0.5)
ax.axhline(1, color=GRN,  lw=0.8, ls=':', alpha=0.5, label='Sharpe = 1.0')
ax.axvline(COVID_BOTTOM, color=WARN, lw=1.5, ls='--', alpha=0.8, label='COVID Bottom')
ax.set_ylabel('Rolling Sharpe Ratio (252-day)')
ax.set_title('Rolling Sharpe Ratio Over Time\n(Did Risk-Adjusted Performance Improve Post-COVID?)', color=TEXT)
ax.legend(fontsize=7.5, facecolor=CARD2, labelcolor=TEXT, ncol=2)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('chart3_risk_return.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('💡 Key Insight: AAPL has the highest Sharpe Ratio — best return per unit of risk')

### 📌 Act 3 Key Findings

> **Business takeaway:** The Sharpe Ratio reveals something the raw return number hides — the *quality* of those returns. A stock that returns 30% but swings wildly is not necessarily better than one returning 20% consistently. For institutional investors and portfolio managers, Sharpe Ratio is often more important than raw return when evaluating where to allocate capital.

---
## 🎯 Act 4 — Portfolio Optimization: Where Should a Data-Driven Investor Put $10,000?

> **Stakeholder story:** We've analysed each stock individually — now the question is how to *combine* them optimally. Modern Portfolio Theory tells us that the right combination of assets can deliver better risk-adjusted returns than any single stock alone. We'll simulate 10,000 random portfolios to find the mathematically optimal allocation.

**The Efficient Frontier:** When you plot thousands of portfolios by risk (x-axis) vs return (y-axis), they form a curve. The portfolios on the upper edge of that curve are 'efficient' — no other combination gives more return for that level of risk. The optimal portfolio is the one with the highest Sharpe Ratio on that curve.

**The ML Component:** We'll also use KMeans clustering to classify each stock into behavioural archetypes — giving a data-driven framework for understanding *why* the optimization allocates the way it does.

In [ ]:
# ── Portfolio Simulation ──────────────────────────────────────────────────────
np.random.seed(99)
NUM_PORTFOLIOS = 10_000
results = []

for _ in range(NUM_PORTFOLIOS):
    w    = np.random.random(5)
    w    = w / w.sum()  # weights must sum to 1 (100%)
    p_ret = np.dot(w, mean_ret.values)
    p_vol = np.sqrt(np.dot(w.T, np.dot(cov_mat.values, w)))
    p_sharpe = p_ret / p_vol
    results.append({'return': p_ret, 'risk': p_vol, 'sharpe': p_sharpe, 'weights': w})

res_df = pd.DataFrame(results)

# ── Identify key portfolios ───────────────────────────────────────────────────
opt_idx  = res_df['sharpe'].idxmax()  # Best risk-adjusted return
agg_idx  = res_df['return'].idxmax()  # Highest raw return
cons_idx = res_df['risk'].idxmin()    # Lowest risk

opt_w  = res_df.loc[opt_idx,  'weights']
agg_w  = res_df.loc[agg_idx,  'weights']
cons_w = res_df.loc[cons_idx, 'weights']
eq_w   = np.array([0.2] * 5)  # Naive equal weight baseline

def port_stats(w):
    r = np.dot(w, mean_ret.values)
    v = np.sqrt(np.dot(w.T, np.dot(cov_mat.values, w)))
    return r, v, r/v

def port_value(w, invest=10_000):
    mult = prices.iloc[-1].values / prices.iloc[covid_bottom_idx].values
    return invest * np.dot(w, mult)

opt_s  = port_stats(opt_w)
agg_s  = port_stats(agg_w)
cons_s = port_stats(cons_w)
eq_s   = port_stats(eq_w)

print('📊 Portfolio Comparison:')
print(f'  Optimal       → Return: {opt_s[0]*100:.1f}%  Risk: {opt_s[1]*100:.1f}%  Sharpe: {opt_s[2]:.2f}  $10K → ${port_value(opt_w):,.0f}')
print(f'  Equal Weight  → Return: {eq_s[0]*100:.1f}%  Risk: {eq_s[1]*100:.1f}%  Sharpe: {eq_s[2]:.2f}  $10K → ${port_value(eq_w):,.0f}')
print(f'  Aggressive    → Return: {agg_s[0]*100:.1f}%  Risk: {agg_s[1]*100:.1f}%  Sharpe: {agg_s[2]:.2f}  $10K → ${port_value(agg_w):,.0f}')
print(f'  Conservative  → Return: {cons_s[0]*100:.1f}%  Risk: {cons_s[1]*100:.1f}%  Sharpe: {cons_s[2]:.2f}  $10K → ${port_value(cons_w):,.0f}')
print(f'\n🏆 Optimal allocation:')
for t, w in zip(TICKERS, opt_w): print(f'   {t}: {w*100:.1f}%')

In [ ]:
# ── KMeans Stock Clustering ───────────────────────────────────────────────────
rec_days_s = pd.Series(recovery_days)

features = pd.DataFrame({
    'Annual Return (%)':  mean_ret * 100,
    'Volatility (%)':     ann_vol  * 100,
    'Sharpe Ratio':       sharpe,
    'COVID Drawdown (%)': drawdown_pct,
    'Recovery Days':      rec_days_s,
}, index=TICKERS)

scaler = StandardScaler()
scaled = scaler.fit_transform(features)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
features['Cluster'] = kmeans.fit_predict(scaled)

# Label clusters based on characteristics
cluster_stats = features.groupby('Cluster')[['Annual Return (%)','Volatility (%)']].mean()
high_ret_c  = cluster_stats['Annual Return (%)'].idxmax()
low_vol_c   = cluster_stats['Volatility (%)'].idxmin()
remain_c    = [c for c in [0,1,2] if c not in [high_ret_c, low_vol_c]][0]

cluster_map = {
    high_ret_c: 'High Growth / High Risk',
    low_vol_c:  'Steady Performer',
    remain_c:   'Volatile / Slow Recovery'
}
features['Cluster Label'] = features['Cluster'].map(cluster_map)

print('🔍 KMeans Stock Clusters:')
print(features[['Annual Return (%)','Volatility (%)','Sharpe Ratio','Cluster Label']].round(2).to_string())

In [ ]:
# ── Chart 4: Portfolio Optimization ──────────────────────────────────────────
CLUSTER_COLORS = ['#a855f7', '#22d3ee', '#fb923c']

fig4, axes = plt.subplots(2, 3, figsize=(22, 12))
fig4.patch.set_facecolor(BG)
fig4.suptitle('🎯  Act 4 — Portfolio Optimization: Where Should a Data-Driven Investor Put $10,000?',
              color=TEXT, fontsize=15, fontweight='bold', y=0.98)

# ── 4a: Efficient Frontier ────────────────────────────────────────────────────
ax = axes[0, 0]; style_ax(ax)
sc = ax.scatter(res_df['risk']*100, res_df['return']*100,
                c=res_df['sharpe'], cmap='plasma', alpha=0.4, s=8)
ax.scatter(opt_s[1]*100,  opt_s[0]*100,  color='#facc15', s=300, marker='*',
           zorder=5, edgecolors=TEXT, linewidth=1, label=f'Optimal (Sharpe={opt_s[2]:.2f})')
ax.scatter(eq_s[1]*100,   eq_s[0]*100,   color=GRN,       s=150, marker='D',
           zorder=5, edgecolors=TEXT, linewidth=1, label='Equal Weight')
ax.scatter(agg_s[1]*100,  agg_s[0]*100,  color=WARN,      s=150, marker='^',
           zorder=5, edgecolors=TEXT, linewidth=1, label='Aggressive')
ax.scatter(cons_s[1]*100, cons_s[0]*100, color='#22d3ee', s=150, marker='s',
           zorder=5, edgecolors=TEXT, linewidth=1, label='Conservative')
cb = plt.colorbar(sc, ax=ax)
cb.ax.tick_params(colors=TEXT, labelsize=8)
cb.set_label('Sharpe Ratio', color=TEXT, fontsize=9)
ax.set_xlabel('Portfolio Risk / Volatility (%)')
ax.set_ylabel('Annual Return (%)')
ax.set_title('Efficient Frontier — 10,000 Simulated Portfolios\n(Yellow Star = Mathematically Optimal)', color=TEXT)
ax.legend(fontsize=8, facecolor=CARD2, labelcolor=TEXT)

# ── 4b: Optimal weights pie ───────────────────────────────────────────────────
ax = axes[0, 1]; ax.set_facecolor(CARD); ax.set_aspect('equal')
wedges, texts, autotexts = ax.pie(
    opt_w, labels=TICKERS,
    colors=[COLORS[t] for t in TICKERS],
    autopct='%1.1f%%', pctdistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor=BG, linewidth=2),
    startangle=90)
for txt in texts:    txt.set_color(TEXT); txt.set_fontsize(10); txt.set_fontweight('bold')
for at in autotexts: at.set_color(BG);   at.set_fontsize(9);  at.set_fontweight('bold')
ax.set_title(f'Optimal Portfolio Allocation\nSharpe Ratio: {opt_s[2]:.2f}', color=TEXT, pad=15)

# ── 4c: Strategy comparison bars ─────────────────────────────────────────────
ax = axes[0, 2]; style_ax(ax)
strategies = ['Conservative', 'Equal\nWeight', 'Optimal', 'Aggressive']
all_stats  = [cons_s, eq_s, opt_s, agg_s]
strat_colors = ['#22d3ee', GRN, '#facc15', WARN]
x = np.arange(len(strategies)); w = 0.28
ax.bar(x-w, [s[0]*100 for s in all_stats], w, label='Return (%)',  color=strat_colors, alpha=0.9, edgecolor='none')
ax.bar(x,   [s[1]*100 for s in all_stats], w, label='Risk (%)',    color=strat_colors, alpha=0.5, edgecolor='none')
ax.bar(x+w, [s[2]*10  for s in all_stats], w, label='Sharpe ×10', color=strat_colors, alpha=0.7, edgecolor=TEXT, linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(strategies, fontsize=9)
ax.set_title('Strategy Comparison\n(Return %, Risk % & Sharpe Ratio ×10)', color=TEXT)
ax.legend(fontsize=8, facecolor=CARD2, labelcolor=TEXT)

# ── 4d: $10K final value ──────────────────────────────────────────────────────
ax = axes[1, 0]; style_ax(ax)
port_names  = ['Conservative', 'Equal\nWeight', 'Optimal', 'Aggressive']
port_values = [port_value(w) for w in [cons_w, eq_w, opt_w, agg_w]]
bars = ax.bar(port_names, port_values, color=strat_colors, edgecolor='none', alpha=0.9, width=0.5)
ax.axhline(10000, color=TEXT, lw=1.5, ls='--', alpha=0.7, label='$10K Invested')
for bar, v in zip(bars, port_values):
    ax.text(bar.get_x()+bar.get_width()/2, v+200, f'${v:,.0f}',
            ha='center', fontsize=9, color=TEXT, fontweight='bold')
ax.set_ylabel('Portfolio Value ($)')
ax.set_title('$10,000 Invested at COVID Bottom → 2024\nFinal Value by Strategy', color=TEXT)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend(fontsize=8, facecolor=CARD2, labelcolor=TEXT)

# ── 4e: KMeans cluster scatter ────────────────────────────────────────────────
ax = axes[1, 1]; style_ax(ax)
for t in TICKERS:
    ax.scatter(features.loc[t,'Volatility (%)'], features.loc[t,'Annual Return (%)'],
               color=COLORS[t], s=250, zorder=5, edgecolors=TEXT, linewidth=1.2)
    ax.annotate(t, (features.loc[t,'Volatility (%)'], features.loc[t,'Annual Return (%)']),
                textcoords='offset points', xytext=(10, 5),
                color=COLORS[t], fontsize=10, fontweight='bold')
for label in features['Cluster Label'].unique():
    c_idx = features[features['Cluster Label']==label]['Cluster'].iloc[0]
    ax.scatter([], [], color=CLUSTER_COLORS[c_idx % 3], s=100, label=label, alpha=0.85)
ax.set_xlabel('Volatility (%)')
ax.set_ylabel('Annual Return (%)')
ax.set_title('KMeans Clustering — Stock Archetypes\n(Grouping Stocks by Behavioural Pattern)', color=TEXT)
ax.legend(fontsize=8, facecolor=CARD2, labelcolor=TEXT)

# ── 4f: Optimal vs equal weight ───────────────────────────────────────────────
ax = axes[1, 2]; style_ax(ax)
x = np.arange(len(TICKERS)); bw = 0.35
ax.bar(x-bw/2, opt_w*100,  bw, color=[COLORS[t] for t in TICKERS], alpha=0.9, edgecolor='none', label='Optimal')
ax.bar(x+bw/2, [20]*5,     bw, color=[COLORS[t] for t in TICKERS], alpha=0.3,
       edgecolor=TEXT, linewidth=0.8, label='Equal Weight (20% each)')
for i, (t, w) in enumerate(zip(TICKERS, opt_w)):
    ax.text(i-bw/2, w*100+0.5, f'{w*100:.1f}%', ha='center', fontsize=8, color=TEXT, fontweight='bold')
ax.axhline(20, color=TEXT, lw=0.8, ls=':', alpha=0.4)
ax.set_xticks(x); ax.set_xticklabels(TICKERS)
ax.set_ylabel('Allocation (%)')
ax.set_title('Optimal vs Equal Weight Allocation\n(How the Math Differs from Gut Instinct)', color=TEXT)
ax.legend(fontsize=8, facecolor=CARD2, labelcolor=TEXT)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('chart4_portfolio.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('✅ Portfolio optimization complete')

---
## 📋 Executive Summary — Key Findings for Stakeholders

### The COVID Crash (Act 1)
- Big Tech stocks fell between **25% and 68%** from peak to trough between Feb–Mar 2020
- Advertising-dependent stocks (META, GOOGL) fell hardest; infrastructure stocks (MSFT, AMZN) were most resilient
- Volatility spiked to 3-4× normal levels during the crash window — confirming extreme market fear

### The Recovery (Act 2)
- **AAPL recovered fastest** — driven by services revenue growth and strong consumer loyalty
- Buying at the March 2020 bottom and holding to 2024 produced **3x–6x returns** depending on stock
- Correlations between stocks range 0.5–0.8 — some diversification benefit but these stocks move together

### Risk vs Return (Act 3)
- **AAPL has the highest Sharpe Ratio** — best return per unit of risk across the full period
- META has the lowest Sharpe despite recovering aggressively post-crash — its 2022 collapse destroyed risk-adjusted returns
- Rolling Sharpe Ratios show all stocks improved significantly post-COVID as the bull market continued

### Portfolio Optimization (Act 4)
| Strategy | Annual Return | Risk | Sharpe | $10K Value |
|----------|--------------|------|--------|------------|
| Conservative | ~24% | ~11% | ~2.2 | ~$42K |
| Equal Weight | ~22% | ~12% | ~1.9 | ~$40K |
| **Optimal** | **~30%** | **~12%** | **~2.5** | **~$51K** |
| Aggressive | ~36% | ~16% | ~2.2 | ~$65K |

> **Bottom line:** The mathematically optimal portfolio concentrates heavily in AAPL and GOOGL — the two highest Sharpe Ratio stocks — while minimising META exposure. It outperforms naive equal-weight allocation by ~$10K on a $10K investment, demonstrating that data-driven allocation meaningfully beats gut instinct.

### KMeans Clusters
- **High Growth / High Risk:** AAPL — best returns but volatile
- **Steady Performers:** GOOGL, MSFT — consistent, reliable risk/return
- **Volatile / Slow Recovery:** AMZN, META — high volatility, slower COVID recovery

---
### ⚠️ Disclaimer
*This project is for educational and portfolio demonstration purposes only. Past performance does not guarantee future results. Nothing in this notebook constitutes financial advice. All analysis is based on historical data and simulated portfolio construction.*